# Synthetic grid v5

All four axes from the task spec, plus the null-edit control the reviewer asked for.

| axis | levels |
|---|---|
| size | small 47px, medium 60px, large 74px |
| location | upper / mid / lower x left / right |
| overlap | medial (toward heart, spine, hilum) / lateral (clear lung) |
| contrast | 3 levels derived post-hoc, no generation needed |

`3 sizes x 3 zones x 2 sides x 2 overlap = 36 cells`, plus 6 null cells, x 11 chests
= **462 images, ~3.2 h**. With contrast that is 108 attribute combinations.

### How location and overlap are kept separate

Both are positional, so they are made orthogonal: **location is the vertical third** of the
detected lung, **overlap is the horizontal position**. Medial anchors just inside the lung
edge facing the mediastinum, where the heart, spine and hilar vessels overlap the lung in
projection; lateral sits out in clear field. Every site records `structure_density`, so
the label has a measurement behind it rather than being a claim.

### What earlier failures forced into this design

- **F0** — masks are specified in pixels and `make_mask` **raises** instead of clamping.
  v3's `if x1 <= x0: x1 = x0+20` turned 89 impossible boxes into slivers that no
  downstream check could see.
- **F3** — every position derives from that chest's own detected lungs. The old grid
  reused fixed coordinates on all 50 chests.
- **F14** — RadEdit is architecturally locked to 512px, floor 47px. All three sizes are
  **masses** (32-50mm), not nodules. State this in Methods.
- **F15** — prompt does not control lesion type, so it is held constant. But it must name
  an anatomical location: three-word prompts painted nothing across 5 seeds.
- **F16** — edit strength varies ~4x between chests, so `edit_inside` is recorded but
  never gated on. `edit_norm` divides it by that chest's own local texture.

### Order

Run **1-12**, then stop. **Cell 12 is the gate**: if medial sites are not reliably denser
than lateral, the overlap axis is null and there is no point starting the 3-hour run.

## 1 - Install and authenticate

In [ ]:
from google.colab import auth
auth.authenticate_user()

import google.auth, googleapiclient.discovery
creds, _ = google.auth.default()
svc = googleapiclient.discovery.build('drive', 'v3', credentials=creds)
print(svc.about().get(fields='user').execute()['user'])

In [ ]:
!pip -q install -U diffusers transformers accelerate huggingface-hub SimpleITK

import os, shutil
from google.colab import userdata, drive
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

# A real mount has this marker file. A plain directory does not -- which is what the
# cleanup cell accidentally created, blocking the mount and then masking it.
def drive_is_mounted():
    return os.path.ismount('/content/drive') or os.path.exists('/content/drive/MyDrive/.file-revisions-by-id')

if not drive_is_mounted():
    if os.path.exists('/content/drive'):
        os.system('fusermount -u /content/drive 2>/dev/null')
        shutil.rmtree('/content/drive', ignore_errors=True)
    drive.mount('/content/drive')

assert os.path.isdir('/content/drive/MyDrive'), 'mount failed'
print('mounted. MyDrive contains:', sorted(os.listdir('/content/drive/MyDrive'))[:15])

## 2 - Imports

In [ ]:
import json, glob, traceback, shutil, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd, cv2, SimpleITK as sitk, torch, torchvision
from PIL import Image, ImageDraw, ImageOps
from scipy import ndimage, stats
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as TF
import matplotlib.pyplot as plt, matplotlib.patches as patches

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
assert DEVICE == 'cuda', 'needs a GPU -- Runtime > Change runtime type'

## 3 - RadEdit

In [ ]:
from transformers import AutoModel, AutoTokenizer
from diffusers import (AutoencoderKL, DDIMScheduler, DiffusionPipeline,
                       StableDiffusionPipeline, UNet2DConditionModel)

unet = UNet2DConditionModel.from_pretrained('microsoft/radedit', subfolder='unet')
vae  = AutoencoderKL.from_pretrained('stabilityai/sdxl-vae')
te   = AutoModel.from_pretrained('microsoft/BiomedVLP-BioViL-T', trust_remote_code=True)
tok  = AutoTokenizer.from_pretrained('microsoft/BiomedVLP-BioViL-T',
                                     model_max_length=128, trust_remote_code=True)
sched = DDIMScheduler(beta_schedule='linear', clip_sample=False,
                      prediction_type='epsilon', timestep_spacing='trailing',
                      steps_offset=1)

gen_pipe = StableDiffusionPipeline(
    vae=vae, text_encoder=te, tokenizer=tok, unet=unet, scheduler=sched,
    safety_checker=None, requires_safety_checker=False, feature_extractor=None).to(DEVICE)
PIPE = DiffusionPipeline.from_pipe(gen_pipe, custom_pipeline='microsoft/radedit', trust_remote_code=True)
print('radedit loaded')

## 4 - Detector

`box_score_thresh=0.0` so nothing the 0.05 cutoff would hide is lost, and **every box is
saved to JSON**. A threshold change or a FROC curve then never requires rerunning anything.

In [ ]:
import glob, os
from pathlib import Path

T = Path('/content/drive/MyDrive/Algoverse/Teammates')
print('exists:', T.exists())
if T.exists():
    for p in sorted(T.rglob('*')):
        if p.is_file() and p.suffix.lower() in ('.pth','.pt','.ckpt','.zip','.csv','.npz'):
            print(f'  {p.relative_to(T)}   ({p.stat().st_size/1e6:.0f} MB)')
        elif p.is_dir():
            print(f'  {p.relative_to(T)}/   ({len(list(p.glob("*")))} items)')
else:
    print('MyDrive/Algoverse contents:')
    for p in sorted(Path('/content/drive/MyDrive/Algoverse').iterdir()):
        print('  ', p.name, '(dir)' if p.is_dir() else '')

In [ ]:
ck = glob.glob('/content/drive/MyDrive/**/baseline1_checkpoint.pth', recursive=True)
assert ck, 'no checkpoint found'
DET_SIZE, SCORE_MIN = 800, 0.05

DET = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights=None, weights_backbone=None,
    box_score_thresh=0.0, box_detections_per_img=300)
DET.roi_heads.box_predictor = FastRCNNPredictor(
    DET.roi_heads.box_predictor.cls_score.in_features, 2)
_st = torch.load(ck[0], map_location=DEVICE)
DET.load_state_dict(_st['model'] if 'model' in _st else _st)
DET = DET.eval().to(DEVICE)

@torch.no_grad()
def detect_all(pil, size=DET_SIZE):
    im = pil.convert('RGB').resize((size, size), Image.LANCZOS)
    o = DET([TF.to_tensor(im).to(DEVICE)])[0]
    b, s = o['boxes'].cpu().numpy()/size, o['scores'].cpu().numpy()
    k = s >= SCORE_MIN
    return b[k], s[k]

def best_at(boxes, scores, t):
    """Highest score whose box CENTRE falls in the lesion box -- the project's matcher."""
    return float(max((ss for bb, ss in zip(boxes, scores)
                      if t[0] <= (bb[0]+bb[2])/2 <= t[2]
                      and t[1] <= (bb[1]+bb[3])/2 <= t[3]), default=0.0))

print('detector loaded from', ck[0])
print('internal transform min_size =', DET.transform.min_size)

## 5 - Configuration

The only cell you should need to edit.

In [ ]:
DRIVE_MHA = Path('/content/drive/MyDrive/Algoverse/data/node21/images')
OUT       = Path('/content/grid_v5')
DRIVE     = Path('/content/drive/MyDrive/Algoverse/grid_v5_runs')
for d in ['images','masks','backgrounds','detections']:
    (OUT/d).mkdir(parents=True, exist_ok=True)
DRIVE.mkdir(parents=True, exist_ok=True)

SIZE, SKIP_RATIO, GUIDANCE, STEPS = 512, 0.3, 7.5, 200
N_CHESTS = 14          # was 8; +6 chests = +6 per cell
CHECKPOINT_EVERY = 1

SIZES = {'small': 47, 'medium': 60, 'large': 74}     # F14: all masses, 32-50mm
ZONES = {'upper': 0.22, 'mid': 0.50, 'lower': 0.78}  # LOCATION axis
SIDES = ['left', 'right']

# OVERLAP axis -- a density GRADIENT, not two extremes. The n=1 pilot showed RadEdit
# paints nothing above structure_density ~0.85 (0/15 dense sites produced a lesion),
# so the extremes gave one usable level. Three percentiles put a level near that
# threshold, where lesions are painted but obscured.
OVERLAPS   = ['clear', 'moderate', 'on_structure']
OV_PCTILE  = {'clear': 10, 'moderate': 50, 'on_structure': 90}

ANCHOR   = 1.05     # mask radii inset from the lung-box edge
SEP_FRAC = 0.60     # lung-width requirement, in mask diameters
N_CAND   = 21       # horizontal positions scanned per zone

PROMPTS = {'left':  'Right upper lobe pulmonary nodule',
           'right': 'Left upper lobe pulmonary nodule'}

# 'null_edit' not 'null' -- pandas parses the string "null" as NaN on read
NULL_PROMPTS = {'left':  'Normal right lung parenchyma, no focal abnormality',
                'right': 'Normal left lung parenchyma, no focal abnormality'}
NULL_ARM = dict(sizes=['large'], overlaps=['clear'])

MAX_MASK = max(SIZES.values())
n_cells  = len(SIZES)*len(ZONES)*len(OVERLAPS)*len(SIDES)
n_null   = len(NULL_ARM['sizes'])*len(NULL_ARM['overlaps'])*len(ZONES)*len(SIDES)
print(f'{n_cells} lesion + {n_null} null = {n_cells+n_null} per chest')
print(f'x {N_CHESTS} chests = {(n_cells+n_null)*N_CHESTS} images'
      f'  (~{(n_cells+n_null)*N_CHESTS*25/3600:.1f} h)')
print(f'lesion grid x3 contrast levels post-hoc = {n_cells*3} attribute combinations')
print(f'marginal n per size level: {n_cells*N_CHESTS//len(SIZES)}')

## 6 - Stage the .mha files on local disk

Reading them one at a time over the Drive mount is what hung for 22 minutes earlier - a
stalled FUSE read blocks inside C, where `KeyboardInterrupt` cannot reach it. One bulk copy
removes that risk and speeds up the whole run.

In [ ]:
LOCAL   = Path('/content/node21'); LOCAL.mkdir(parents=True, exist_ok=True)
N_STAGE = 60          # ~5x N_CHESTS, headroom for the ~25% rejection rate

names = sorted(p.name for p in DRIVE_MHA.glob('*.mha'))[:N_STAGE]
assert names, f'no .mha under {DRIVE_MHA}'
print(f'staging {len(names)} of {len(list(DRIVE_MHA.glob("*.mha")))} files...')

t0 = time.time()
for i, n in enumerate(names):
    dst = LOCAL/n
    if not dst.exists():
        subprocess.run(['cp', str(DRIVE_MHA/n), str(dst)], check=True)
    if i % 10 == 0:
        print(f'  {i}/{len(names)}  ({time.time()-t0:.0f}s)')

MHA_DIR = LOCAL
mb = sum(p.stat().st_size for p in LOCAL.glob('*.mha'))/1e6
print(f'\ndone in {time.time()-t0:.0f}s - {len(list(LOCAL.glob("*.mha")))} files, '
      f'{mb:.0f} MB. MHA_DIR now points at local disk.')

## 7 - Preprocessing

Crop square rather than squash - aspect ratios in NODE21 run 0.82 to 1.22, so resizing
straight to square stretched the most portrait chests by over 20%. Clip to the 1st-99th
percentile rather than min/max, because several images saturate at both ends and min-max
anchors on those.

In [ ]:
def load_chest(path, size=SIZE):
    a = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(np.float32)
    a = a.squeeze() if a.ndim == 3 else a
    oh, ow = a.shape
    s = min(oh, ow); x0, y0 = (ow-s)//2, (oh-s)//2
    a = a[y0:y0+s, x0:x0+s]
    lo, hi = np.percentile(a, [1, 99])
    a = np.clip((a-lo)/(hi-lo+1e-8), 0, 1)
    a = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)) \
           .apply((a*255).astype(np.uint8)).astype(np.float32)/255.0
    a = np.clip(cv2.resize(a, (size,size), interpolation=cv2.INTER_AREA), 0, 1)
    prov = dict(orig_w=ow, orig_h=oh, crop_x0=x0, crop_y0=y0, crop_side=s,
                scale=round(size/s, 5))
    return Image.fromarray((a*255).astype(np.uint8)).convert('RGB'), a, prov

## 8 - Lung detection

An X-ray is dark where there is air, so the lungs are the two big dark regions inside the
chest. Blobs touching the border are the black area outside the body, not lung.

Chests differ in exposure, so `find_lungs_auto` tries several thresholds and keeps the
largest lungs that pass every check. It also **rejects chests whose lungs are too narrow to
separate medial from lateral** - on those the overlap axis would silently measure nothing.

In [ ]:
def find_lungs(img, dark_percentile=35, min_area_frac=0.02):
    h, w = img.shape
    a = (img-img.min())/(img.max()-img.min()+1e-8)
    dark = a < np.percentile(a, dark_percentile)
    dark = ndimage.binary_closing(ndimage.binary_opening(dark, np.ones((5,5))),
                                  np.ones((9,9)))
    lab, n = ndimage.label(dark)
    if n == 0: return None
    border = (set(lab[0,:]) | set(lab[-1,:]) | set(lab[:,0]) | set(lab[:,-1]))
    border.discard(0)
    cands = []
    for i in range(1, n+1):
        if i in border: continue
        ys, xs = np.where(lab == i)
        if len(ys) < min_area_frac*h*w: continue
        cands.append(dict(area=len(ys), x0=xs.min(), x1=xs.max(),
                          y0=ys.min(), y1=ys.max(), cx=xs.mean()))
    if len(cands) < 2: return None
    cands.sort(key=lambda c: -c['area'])
    two = sorted(cands[:2], key=lambda c: c['cx'])
    return tuple((c['x0']/w, c['y0']/h, c['x1']/w, c['y1']/h) for c in two)


def check_lungs(l, r, max_mask_px=None, size=SIZE, sep_frac=SEP_FRAC):
    bad = []
    lw, rw, lh, rh = l[2]-l[0], r[2]-r[0], l[3]-l[1], r[3]-r[1]
    if l[2] > r[0]+0.05: bad.append('lung boxes overlap')
    if not 0.10 < lw < 0.45 or not 0.10 < rw < 0.45: bad.append(f'widths {lw:.2f},{rw:.2f}')
    if not 0.20 < lh < 0.80 or not 0.20 < rh < 0.80: bad.append(f'heights {lh:.2f},{rh:.2f}')
    if max(lw,rw)/max(min(lw,rw),1e-6) > 2.0: bad.append('one lung twice the other')
    if abs(l[1]-r[1]) > 0.20: bad.append('lung tops very different')
    if l[0] < 0.02 or r[2] > 0.98: bad.append('box touches image edge')
    if max_mask_px is not None:
        need = 2*ANCHOR*(max_mask_px/2) + sep_frac*max_mask_px
        for nm, box in [('left', l), ('right', r)]:
            wpx = (box[2]-box[0])*size
            if wpx < need:
                sep = wpx - 2*ANCHOR*(max_mask_px/2)
                bad.append(f'{nm} lung {wpx:.0f}px too narrow: separation {sep:.0f}px, '
                           f'need {sep_frac*max_mask_px:.0f}')
    return bad


def find_lungs_auto(img, max_mask_px=None, percentiles=(30, 35, 40, 45, 50)):
    """Returns (left, right, percentile_used) or None."""
    best = None
    for p in percentiles:
        res = find_lungs(img, dark_percentile=p)
        if res is None: continue
        if check_lungs(*res, max_mask_px=max_mask_px): continue
        area = sum((b[2]-b[0])*(b[3]-b[1]) for b in res)
        if best is None or area > best[0]:
            best = (area, res, p)
    return (best[1][0], best[1][1], best[2]) if best else None

## 9 - Geometry

`make_mask` **raises** rather than shrinking, shifting or clamping. v3's
`if x1 <= x0: x1 = x0 + 20` is why 89 masks came out one pixel wide and stayed invisible to
every check downstream (F0).

In [ ]:
_Y, _X = np.mgrid[0:SIZE, 0:SIZE]      # precomputed once; the search calls this a lot

def make_mask(cx, cy, mask_px, size=SIZE):
    """Raises rather than clamping. v3's `if x1 <= x0: x1 = x0+20` is why 89 masks came
    out one pixel wide and stayed invisible to every check downstream (F0)."""
    if mask_px < 47: raise ValueError(f'{mask_px}px below the 47px latent floor')
    r  = mask_px/2
    bx = (cx*size-r, cy*size-r, cx*size+r, cy*size+r)
    if not (0 <= bx[0] < bx[2] <= size and 0 <= bx[1] < bx[3] <= size):
        raise ValueError(f'mask {tuple(round(v,1) for v in bx)} does not fit')
    m = Image.new('L', (size,size), 0); ImageDraw.Draw(m).ellipse(list(bx), fill=255)
    achieved = round(bx[2]-bx[0], 2)
    assert abs(achieved-mask_px) < 1e-6, 'mask width drifted from request'
    return m, tuple(v/size for v in bx), achieved


def fits_lung(mbox, lung, tol=0.0):
    return (mbox[0] >= lung[0]-tol and mbox[1] >= lung[1]-tol
            and mbox[2] <= lung[2]+tol and mbox[3] <= lung[3]+tol)


def density_at(arr01, cx, cy, mask_px, lung, size=SIZE):
    """Brightness at the site relative to the median of the whole lung field.
    >1 means more tissue is superimposed there. The pilot put the painting threshold
    at roughly 0.85."""
    a = arr01*255.0; r = mask_px/2
    d = np.sqrt((_X-cx*size)**2 + (_Y-cy*size)**2)
    core = d <= r*0.7
    if core.sum() < 20: return None
    lx0, ly0, lx1, ly1 = [int(v*size) for v in lung]
    return float(a[core].mean() / (np.median(a[ly0:ly1, lx0:lx1]) + 1e-6))


def overlap_sites(arr01, lung, zone, mask_px, size=SIZE, n_cand=N_CAND):
    """
    Scan horizontally across the lung at this zone's height and return one site at each
    density percentile in OV_PCTILE.

    Selection is by DENSITY, not by position -- the pilot disproved the medial/lateral
    assumption (densest is lateral at the apex, medial at the base), and percentiles keep
    a real gradient even on chests where the density range is narrow. `density_spread` is
    returned so uninformative cells can be filtered afterwards rather than silently
    contributing noise.
    """
    x0, y0, x1, y1 = lung
    r  = (mask_px/2)/size
    cy = y0 + ZONES[zone]*(y1-y0)
    lo, hi = x0 + ANCHOR*r, x1 - ANCHOR*r
    if hi <= lo: return None

    cands = []
    for cx in np.linspace(lo, hi, n_cand):
        try:
            _, mbox, _ = make_mask(cx, cy, mask_px)
        except ValueError:
            continue
        if not fits_lung(mbox, lung): continue
        dn = density_at(arr01, cx, cy, mask_px, lung)
        if dn is not None:
            cands.append((cx, dn))
    if len(cands) < len(OVERLAPS): return None

    cands.sort(key=lambda t: t[1])                 # ascending density
    dens = [c[1] for c in cands]
    spread = dens[-1] - dens[0]

    out = {}
    for name, pct in OV_PCTILE.items():
        i = int(round((pct/100) * (len(cands)-1)))
        cx, dn = cands[i]
        out[name] = (cx, cy, dn, spread)
    return out

## 10 - Measurement

`edit_inside` is **recorded, never gated on** - and it is not comparable between chests
(F16): 9.44 on one chest detected at 0.966 while 7.93 on another detected at 0.000.
`edit_norm` divides by that chest's own local texture; `cnr` and `structure_density` are
the cross-chest comparable measures.

In [ ]:
def edit_inside(before, after, mask):
    b = np.asarray(before.convert('L'), np.float32)
    a = np.asarray(after.convert('L'),  np.float32)
    return float(np.abs(a-b)[np.asarray(mask, bool)].mean())


def site_stats(arr01, cx, cy, mask_px, lung, size=SIZE):
    """Local texture, baseline offset, and structure_density -- brightness at the site
    relative to the whole lung field. Density is the objective measure behind the
    medial/lateral label."""
    a = arr01*255.0; r = mask_px/2
    Y, X = np.mgrid[0:size, 0:size]
    d = np.sqrt((X-cx*size)**2 + (Y-cy*size)**2)
    core, ring = d <= r*0.7, (d > r*1.15) & (d <= r*1.8)
    if ring.sum() < 50 or core.sum() < 20: return None
    lx0, ly0, lx1, ly1 = [int(v*size) for v in lung]
    field = a[ly0:ly1, lx0:lx1]
    density = float(a[core].mean() / (np.median(field) + 1e-6))
    return float(a[ring].std()), float(a[core].mean()-a[ring].mean()), core, ring, density


def conspicuity(after, core, ring, base, lstd):
    """Contrast-to-noise. Detection collapsed between CNR 0.66 and 0.52 in testing."""
    a = np.asarray(after.convert('L'), np.float32)
    return float(((a[core].mean()-a[ring].mean()) - base)/(lstd+1e-6))


def generate(bg, mask, seed, prompt):
    torch.manual_seed(seed)
    return PIPE(prompt, weights=[GUIDANCE], image=bg, edit_mask=mask,
                keep_mask=ImageOps.invert(mask), num_inference_steps=STEPS,
                invert_prompt='', skip_ratio=SKIP_RATIO, output_type='pil')[0]

## 11 - Preflight

Red circles must hug the inner edges either side of the bright central column, green must
sit out in clear lung, and no circle may spill outside its cyan box.

In [ ]:
probe = sorted(p.name for p in MHA_DIR.glob('*.mha'))[:8]
assert probe, f'no .mha in {MHA_DIR} -- did Cell 6 run?'
COL = {'clear': 'lime', 'moderate': 'gold', 'on_structure': 'red'}

fig, ax = plt.subplots(1, len(probe), figsize=(3.6*len(probe), 4.6), squeeze=False)
for a_, nm in zip(ax.ravel(), probe):
    _, arr, _ = load_chest(MHA_DIR/nm)
    a_.imshow(arr, cmap='gray'); a_.axis('off')

    got = find_lungs_auto(arr, max_mask_px=MAX_MASK)
    if got is None:
        raw = find_lungs(arr)
        why = 'no two lung regions' if raw is None else \
              check_lungs(*raw, max_mask_px=MAX_MASK)[0][:30]
        a_.set_title(f'{nm}\nREJECTED: {why}', fontsize=7, color='crimson'); continue

    left, right, pct = got
    spreads = []
    for side, box in zip(SIDES, (left, right)):
        a_.add_patch(patches.Rectangle(
            (box[0]*SIZE, box[1]*SIZE), (box[2]-box[0])*SIZE, (box[3]-box[1])*SIZE,
            fill=False, edgecolor='cyan' if side == 'left' else 'deepskyblue', lw=1.2))
        for zone in ZONES:
            sites = overlap_sites(arr, box, zone, MAX_MASK)
            if sites is None: continue
            for ov, (cx, cy, dn, spread) in sites.items():
                a_.add_patch(patches.Circle((cx*SIZE, cy*SIZE), MAX_MASK/2,
                             fill=False, edgecolor=COL[ov], lw=1.3))
                spreads.append(spread)
    a_.set_title(f'{nm}  pct {pct}\ndensity spread '
                 f'{np.median(spreads):.2f}' if spreads else f'{nm}\nno sites',
                 fontsize=7)

plt.suptitle('green = clear (lowest density)   gold = moderate   red = on_structure '
             '(highest)\nall three must sit inside the cyan lung box', fontsize=10)
plt.tight_layout(); plt.show()

## 12 - Gate: does the overlap axis actually exist?

Geometric separation is not the test. **Medial sites have to be measurably denser**, or
the axis is null and Cell 14 would fail after three hours instead of now.

Two minutes, no GPU. Do not start Cell 13 until this passes.

In [ ]:
rows = []
for nm in sorted(p.name for p in MHA_DIR.glob('*.mha'))[:25]:
    try:
        _, arr, _ = load_chest(MHA_DIR/nm)
    except Exception:
        continue
    got = find_lungs_auto(arr, max_mask_px=MAX_MASK)
    if got is None: continue
    left, right, _ = got
    for side, lung in zip(SIDES, (left, right)):
        lung_mid = (lung[0]+lung[2])/2
        for zone in ZONES:
            sites = overlap_sites(arr, lung, zone, MAX_MASK)
            if sites is None: continue
            for ov, (cx, cy, dn, spread) in sites.items():
                rows.append(dict(chest=Path(nm).stem, side=side, zone=zone, overlap=ov,
                                 density=round(dn,4), spread=round(spread,4),
                                 position='medial' if ((side=='left')==(cx>lung_mid))
                                          else 'lateral'))

P = pd.DataFrame(rows)
P['overlap'] = pd.Categorical(P.overlap, OVERLAPS, ordered=True)
print(f'{P.chest.nunique()} chests, {len(P)} sites\n')
print(P.groupby('overlap', observed=True).density
       .describe()[['count','mean','50%','std']].round(3).to_string())
print('\nby zone:')
print(P.pivot_table(index='zone', columns='overlap', values='density',
                    observed=True).round(3).to_string())
print('\nwhere each level lands:')
print(pd.crosstab(P.overlap, [P.zone, P.position]).to_string())

# the level that matters: does 'moderate' sit near the ~0.85 painting threshold?
print(f'\nmoderate: {(P[P.overlap=="moderate"].density < 0.85).mean():.0%} '
      f'of sites below the 0.85 painting threshold')
print(f'clear:    {(P[P.overlap=="clear"].density < 0.85).mean():.0%}')
print(f'on_struct:{(P[P.overlap=="on_structure"].density < 0.85).mean():.0%}')

w = P.pivot_table(index=['chest','side','zone'], columns='overlap',
                  values='density', observed=True).dropna()
print(f'\nordering holds (clear < moderate < on_structure): '
      f'{((w["clear"] < w["moderate"]) & (w["moderate"] < w["on_structure"])).mean():.0%}')
print(f'density spread per site, median {P.spread.median():.3f}, '
      f'{(P.spread < 0.3).mean():.0%} of sites below 0.3 (too flat to be informative)')

## 13 - The run

Writes after every chest and checkpoints to Drive every `CHECKPOINT_EVERY` chests, so a
disconnect costs at most two chests. Re-running resumes from what is on disk.

In [ ]:
import zipfile, os, pandas as pd
from pathlib import Path

assert os.path.ismount('/content/drive'), 'Drive not mounted -- re-run Cell 1'

SRC = Path('/content/drive/MyDrive/Algoverse/grid_v5b_runs')
assert SRC.exists(), f'{SRC} not found'
print('in Drive:', [(p.name, f'{p.stat().st_size/1e6:.1f}MB') for p in sorted(SRC.iterdir())])

OUT.mkdir(parents=True, exist_ok=True)

# images/masks/backgrounds/detections come from the zip
zips = sorted(SRC.glob('*.zip'), key=lambda p: p.stat().st_mtime)
assert zips, f'no zip in {SRC}'
with zipfile.ZipFile(zips[-1]) as z:
    z.extractall(OUT)
print(f'unzipped {zips[-1].name}')

# the loose CSVs in Drive may be newer than the copies inside the zip -- prefer them
for name in ['grid_v5.csv', 'skipped.csv']:
    src = SRC/name
    if src.exists():
        dst = OUT/name
        if not dst.exists() or src.stat().st_mtime > dst.stat().st_mtime:
            import shutil; shutil.copy(src, dst)
            print(f'  {name}: took the Drive copy (newer)')

_d = pd.read_csv(OUT/'grid_v5.csv', keep_default_na=False, na_values=[''])
n_img = len(list((OUT/'images').glob('*.png')))
print(f'\nrestored {n_img} images, {len(_d)} rows, {_d.chest.nunique()} chests')
print('chests:', sorted(_d.chest.unique()))
assert n_img == len(_d), f'{len(_d)} rows but {n_img} images -- resume would be wrong'

In [ ]:
missing = [n for n in ['PIPE','DET','MHA_DIR','OUT','DRIVE','N_CHESTS','SIZES','ZONES',
                       'OVERLAPS','OV_PCTILE','SIDES','PROMPTS','NULL_PROMPTS','NULL_ARM',
                       'MAX_MASK','load_chest','find_lungs_auto','overlap_sites',
                       'make_mask','site_stats','conspicuity','generate','detect_all',
                       'best_at','edit_inside','fits_lung']
           if n not in dir()]
print('MISSING:', missing if missing else 'nothing')
print(f'N_CHESTS={N_CHESTS}  CHECKPOINT_EVERY={CHECKPOINT_EVERY}  '
      f'staged={len(list(MHA_DIR.glob("*.mha")))} mha')

In [ ]:
import shutil
from pathlib import Path

# The previous pilot wrote a 42-column CSV; adding density_spread made it 43, and the
# append reused the old header, so columns shift mid-file. The resume also matched old
# image_ids, so 33 of your 60 images sit at the OLD site positions. Start clean.
OUT   = Path('/content/grid_v5b')
DRIVE = Path('/content/drive/MyDrive/Algoverse/grid_v5b_runs')

if OUT.exists():
    shutil.rmtree(OUT)
for d in ['images','masks','backgrounds','detections']:
    (OUT/d).mkdir(parents=True, exist_ok=True)
DRIVE.mkdir(parents=True, exist_ok=True)
print(f'clean: {OUT}  ->  {DRIVE}')

### Deterministic seeding

Run this before the generation cell. It replaces the salted `hash()` seed with a stable one, and reloads the seeds actually used for the existing grid so those images stay byte-identical.

In [ ]:
import hashlib

def stable_seed(image_id: str) -> int:
    """Deterministic across processes and machines.

    The first version of this notebook used `abs(hash(image_id)) % 2**31`. Python salts
    string hashing per interpreter, so that returns a different value on every run --
    a rerun produced a DIFFERENT grid rather than the same one. md5 is stable.
    """
    return int(hashlib.md5(image_id.encode()).hexdigest()[:8], 16) % (2**31)


# The 12-chest grid in results/grid_v5.csv was generated before this fix, with salted
# seeds. Those seeds are recorded, so loading them reproduces those images exactly while
# anything new gets a deterministic seed.
PRIOR_SEEDS = {}
_prior = Path('results/grid_v5.csv')
if _prior.exists():
    import pandas as _pd
    _p = _pd.read_csv(_prior, keep_default_na=False, na_values=[''])
    PRIOR_SEEDS = dict(zip(_p.image_id, _p.seed))
    print(f'loaded {len(PRIOR_SEEDS)} recorded seeds -- existing images will reproduce exactly')
else:
    print('no prior grid found; all seeds from stable_seed()')

def seed_for(image_id: str) -> int:
    return int(PRIOR_SEEDS.get(image_id, stable_seed(image_id)))

In [ ]:
COLS = ['image_id','chest','arm','side','zone','overlap','position','size','seed',
        'orig_w','orig_h','crop_x0','crop_y0','crop_side','scale',
        'lung_pct','lung_x0','lung_y0','lung_x1','lung_y1','cx','cy',
        'mask_x0','mask_y0','mask_x1','mask_y1','mask_px_requested','mask_px_achieved',
        'edit_inside','local_std','edit_norm','cnr','structure_density','density_spread',
        'det_edited','det_background','gain','n_boxes',
        'coord_space','prompt','skip_ratio','guidance','steps']

csv_path, skip_path = OUT/'grid_v5.csv', OUT/'skipped.csv'
done = set()
if csv_path.exists():
    rec  = set(pd.read_csv(csv_path, keep_default_na=False, na_values=['']).image_id)
    disk = {p.stem for p in (OUT/'images').glob('*.png')} & \
           {p.stem for p in (OUT/'masks').glob('*.png')}
    done = rec & disk
    if rec - disk:
        print(f'{len(rec-disk)} CSV rows have no image -- regenerating those')
print(f'resuming: {len(done)} cells done')

def package(tag):
    z = shutil.make_archive(f'/content/grid_v5_{tag}', 'zip', OUT)
    shutil.copy(z, DRIVE/f'grid_v5_{tag}.zip')
    shutil.copy(OUT/'grid_v5.csv', DRIVE/'grid_v5.csv')
    print(f'  >>> {tag}: {Path(z).stat().st_size/1e6:.1f} MB -> Drive [safe]')

chests = sorted(p.name for p in MHA_DIR.glob('*.mha'))
rows, skipped, used, last, scanned = [], [], 0, 0, 0

for name in chests:
    if used >= N_CHESTS:
        break
    scanned += 1
    stem = Path(name).stem

    try:
        bg_pil, bg_arr, prov = load_chest(MHA_DIR/name)
    except Exception as e:
        skipped.append(dict(chest=stem, reason=f'load failed: {e}')); continue

    got = find_lungs_auto(bg_arr, max_mask_px=MAX_MASK)
    if got is None:
        raw = find_lungs(bg_arr)
        why = 'no two lung regions' if raw is None else \
              '; '.join(check_lungs(*raw, max_mask_px=MAX_MASK))
        skipped.append(dict(chest=stem, reason=f'lung: {why}')); continue
    left, right, lung_pct = got

    used += 1
    bg_pil.save(OUT/'backgrounds'/f'{stem}.png')
    boxes  = dict(zip(SIDES, (left, right)))
    bb, bs = detect_all(bg_pil)              # background detected once per chest
    print(f'[{used}/{N_CHESTS}] {stem}   (scanned {scanned}, pct {lung_pct})')

    for side in SIDES:
        lung = boxes[side]
        lung_mid = (lung[0] + lung[2]) / 2
        for zone in ZONES:
            for size_name, mask_px in SIZES.items():

                sites = overlap_sites(bg_arr, lung, zone, mask_px)
                if sites is None:
                    skipped.append(dict(chest=stem,
                        image_id=f'{stem}_{side}_{zone}_*_{size_name}',
                        reason='could not place three sites in this lung'))
                    continue

                for overlap, (cx, cy, dens_search, spread) in sites.items():
                    image_id = f'{stem}_{side}_{zone}_{overlap}_{size_name}'
                    if image_id in done:
                        continue
                    try:
                        mask, mbox, achieved = make_mask(cx, cy, mask_px)
                        if not fits_lung(mbox, lung):
                            raise ValueError('mask outside the detected lung')
                        st = site_stats(bg_arr, cx, cy, mask_px, lung)
                        if st is None:
                            raise ValueError('site too close to the edge')
                    except Exception as e:
                        skipped.append(dict(chest=stem, image_id=image_id, reason=str(e)))
                        print(f'    skip {image_id}: {e}'); continue

                    lstd, base, core, ring, density = st
                    position = 'medial' if ((side == 'left') == (cx > lung_mid)) \
                               else 'lateral'
                    seed  = seed_for(image_id)
                    det_b = best_at(bb, bs, mbox)

                    try:
                        edited = generate(bg_pil, mask, seed, PROMPTS[side])
                    except Exception:
                        skipped.append(dict(chest=stem, image_id=image_id,
                            reason='generation raised: ' + traceback.format_exc(limit=2)))
                        continue

                    edited.save(OUT/'images'/f'{image_id}.png')
                    mask.save(OUT/'masks'/f'{image_id}.png')
                    eb, es = detect_all(edited)
                    json.dump({'edited':     {'boxes': eb.tolist(), 'scores': es.tolist()},
                               'background': {'boxes': bb.tolist(), 'scores': bs.tolist()}},
                              open(OUT/'detections'/f'{image_id}.json', 'w'))

                    ei    = edit_inside(bg_pil, edited, mask)
                    det_e = best_at(eb, es, mbox)

                    rows.append(dict(
                        image_id=image_id, chest=stem, arm='lesion', side=side, zone=zone,
                        overlap=overlap, position=position, size=size_name, seed=seed,
                        **prov, lung_pct=lung_pct,
                        lung_x0=round(lung[0],4), lung_y0=round(lung[1],4),
                        lung_x1=round(lung[2],4), lung_y1=round(lung[3],4),
                        cx=round(cx,4), cy=round(cy,4),
                        mask_x0=round(mbox[0],4), mask_y0=round(mbox[1],4),
                        mask_x1=round(mbox[2],4), mask_y1=round(mbox[3],4),
                        mask_px_requested=mask_px, mask_px_achieved=achieved,
                        edit_inside=round(ei,2), local_std=round(lstd,2),
                        edit_norm=round(ei/(lstd+1e-6),4),
                        cnr=round(conspicuity(edited, core, ring, base, lstd),4),
                        structure_density=round(density,4),
                        density_spread=round(spread,4),
                        det_edited=round(det_e,4), det_background=round(det_b,4),
                        gain=round(det_e-det_b,4), n_boxes=int(len(es)),
                        coord_space='fraction_of_image', prompt=PROMPTS[side],
                        skip_ratio=SKIP_RATIO, guidance=GUIDANCE, steps=STEPS))
                    print(f'    {image_id:<50} edit {ei:6.2f}  dens {density:.2f}  '
                          f'det {det_e:.3f}')

                    # ---- null-edit control: same site, mask and seed, normal-tissue prompt
                    if (size_name in NULL_ARM['sizes']
                            and overlap in NULL_ARM['overlaps']):
                        nid = f'{image_id}__null'
                        if nid not in done:
                            try:
                                nedit = generate(bg_pil, mask, seed, NULL_PROMPTS[side])
                            except Exception:
                                skipped.append(dict(chest=stem, image_id=nid,
                                    reason='null generation raised: '
                                           + traceback.format_exc(limit=2)))
                            else:
                                nedit.save(OUT/'images'/f'{nid}.png')
                                mask.save(OUT/'masks'/f'{nid}.png')
                                nb, ns = detect_all(nedit)
                                json.dump(
                                    {'edited':     {'boxes': nb.tolist(),
                                                    'scores': ns.tolist()},
                                     'background': {'boxes': bb.tolist(),
                                                    'scores': bs.tolist()}},
                                    open(OUT/'detections'/f'{nid}.json', 'w'))
                                nei  = edit_inside(bg_pil, nedit, mask)
                                ndet = best_at(nb, ns, mbox)
                                rows.append({**rows[-1],
                                    'image_id': nid, 'arm': 'null_edit',
                                    'prompt': NULL_PROMPTS[side],
                                    'edit_inside': round(nei, 2),
                                    'edit_norm': round(nei/(lstd+1e-6), 4),
                                    'cnr': round(conspicuity(nedit, core, ring,
                                                             base, lstd), 4),
                                    'det_edited': round(ndet, 4),
                                    'gain': round(ndet - det_b, 4),
                                    'n_boxes': int(len(ns))})
                                print(f'    {nid:<50} edit {nei:6.2f}  '
                                      f'det {ndet:.3f}   [NULL]')

    if rows:
        pd.DataFrame(rows)[COLS].to_csv(csv_path, mode='a',
                                        header=not csv_path.exists(), index=False)
        rows = []
    if skipped:
        pd.DataFrame(skipped).to_csv(skip_path, mode='a',
                                     header=not skip_path.exists(), index=False)
        skipped = []
    if used - last >= CHECKPOINT_EVERY:
        package(f'ckpt{used:03d}'); last = used

print(f'\ndone. {used} chests used, {scanned} scanned.')
package('final')

In [ ]:
from pathlib import Path
for p in [Path('/content/grid_v5b'), Path('/content/grid_v5'), Path('/content/node21')]:
    if p.exists():
        sub = {d.name: len(list(d.glob('*'))) for d in p.iterdir() if d.is_dir()}
        files = [f.name for f in p.iterdir() if f.is_file()]
        print(f'{p}:  {sub}  files={files}')
    else:
        print(f'{p}: does not exist')

In [ ]:
import zipfile, os
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/Algoverse/grid_v5b_runs')
assert os.path.ismount('/content/drive'), 'Drive not mounted'
zips = sorted(DRIVE.glob('*.zip'), key=lambda p: p.stat().st_mtime)
print('in Drive:', [(p.name, f'{p.stat().st_size/1e6:.0f}MB') for p in zips])

if zips:
    OUT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[-1]) as z:
        z.extractall(OUT)
    print(f'restored {zips[-1].name}: '
          f'{len(list((OUT/"images").glob("*.png")))} images')
else:
    print('nothing in Drive -- c0001 will simply regenerate (deterministic, ~25 min)')

## 14 - Verify

Every assertion here corresponds to a way the previous grid was silently wrong. None of
them existed on the old one.

In [ ]:
df = pd.read_csv(OUT/'grid_v5.csv', keep_default_na=False, na_values=[''])
df['overlap'] = pd.Categorical(df.overlap, OVERLAPS, ordered=True)
print(f'{len(df)} rows, {df.chest.nunique()} chests, arms {df.arm.value_counts().to_dict()}')

assert df.image_id.nunique() == len(df), 'duplicate image_id'
assert (df.mask_px_achieved == df.mask_px_requested).all(), 'mask differs from request'
assert (df.mask_px_achieved >= 47).all(), 'below the 47px latent floor'
for c in ['cx','cy','mask_x0','mask_y0','mask_x1','mask_y1']:
    assert df[c].between(0,1).all(), f'{c} outside [0,1] -- wrong coordinate space'
assert ((df.mask_x0 >= df.lung_x0) & (df.mask_y0 >= df.lung_y0)
        & (df.mask_x1 <= df.lung_x1) & (df.mask_y1 <= df.lung_y1)).all(), \
       'a mask sits outside the lung it claims to be in'
assert (df.coord_space == 'fraction_of_image').all(), 'mixed coordinate spaces'

for sub, ext in [('images','png'), ('masks','png'), ('detections','json')]:
    miss = [i for i in df.image_id if not (OUT/sub/f'{i}.{ext}').exists()]
    assert not miss, f'{len(miss)} missing in {sub}: {miss[:5]}'

bad = []
for _, r in df.iterrows():
    m = np.asarray(Image.open(OUT/'masks'/f'{r.image_id}.png'))
    ys, xs = np.nonzero(m)
    if len(xs) == 0: bad.append((r.image_id, 'EMPTY')); continue
    w = xs.max()-xs.min()+1
    if abs(w - r.mask_px_requested) > 2:
        bad.append((r.image_id, f'{w} vs {r.mask_px_requested}'))
assert not bad, f'{len(bad)} masks mismatch: {bad[:5]}'

o = df.groupby('zone').cy.mean()
assert o['upper'] < o['mid'] < o['lower'], 'zone labels do not order vertically'

les = df[df.arm == 'lesion']; nul = df[df.arm == 'null_edit']
dens = les.groupby('overlap', observed=True).structure_density.mean()
print(f'\ndensity by level: {dens.round(3).to_dict()}')
assert dens['clear'] < dens['moderate'] < dens['on_structure'], \
    'density levels are not ordered -- the overlap manipulation is broken'

print('\n--- generation vs detection, by overlap level ---')
print(les.groupby('overlap', observed=True).agg(
    n=('det_edited','size'),
    edit=('edit_inside','mean'),
    density=('structure_density','mean'),
    painted=('edit_inside', lambda s: f'{(s>15).sum()}/{len(s)}'),
    detected=('det_edited', lambda s: f'{(s>0.05).sum()}/{len(s)}'),
    mean_det=('det_edited','mean')).round(3).to_string())

print('\ncells:')
g = les.groupby(['side','zone','overlap','size'], observed=True).size()
print(f'{len(g)} cells, n per cell {g.min()}-{g.max()}')
print(f'background fired pre-edit: {(df.det_background > 0.05).sum()} of {len(df)}')

if len(nul):
    print(f'\nNULL ARM: {len(nul)} images')
    print(f'  fired at the site (>0.05): {(nul.det_edited > 0.05).sum()} of {len(nul)}'
          f'  ({(nul.det_edited > 0.05).mean():.1%})')
    print(f'  mean det {nul.det_edited.mean():.3f}  vs lesion {les.det_edited.mean():.3f}')

print('\nall checks passed')

## 15 - Package and download

The Drive copy needs nobody present; the browser download needs you at the keyboard. Safe
to re-run this cell as many times as it takes.

In [ ]:
import json, shutil, time, os
from pathlib import Path
import pandas as pd

OUT   = Path('/content/grid_v5b')                                   # local scratch
DRIVE = Path('/content/drive/MyDrive/Algoverse/grid_v5b_runs')      # durable copy
ZIP   = Path('/content/grid_v5_final.zip')
STATE = Path('/content/.pkg_state.json')

if not OUT.exists():
    print(f'{OUT} does not exist -- nothing generated in this runtime yet')
else:
    n_img = len(list((OUT/'images').glob('*.png')))
    csv   = OUT/'grid_v5.csv'
    n_row = n_chest = 0
    if csv.exists():
        _d = pd.read_csv(csv, keep_default_na=False, na_values=[''])
        n_row, n_chest = len(_d), _d.chest.nunique()

    prev  = json.load(open(STATE))['n_img'] if STATE.exists() else -1
    stamp = time.strftime('%H:%M:%S')

    if n_img == prev and ZIP.exists():
        # nothing new -- sub-second no-op, safe to spam
        print(f'[{stamp}] nothing new: {n_img} images, {n_chest} chests, '
              f'zip {ZIP.stat().st_size/1e6:.1f} MB already saved')
    else:
        print(f'[{stamp}] packaging {n_img} images / {n_row} rows / {n_chest} chests ...')
        shutil.make_archive('/content/grid_v5_final', 'zip', OUT)
        mb = ZIP.stat().st_size/1e6
        if os.path.ismount('/content/drive'):
            DRIVE.mkdir(parents=True, exist_ok=True)
            shutil.copy(ZIP, DRIVE/'grid_v5_final.zip')
            if csv.exists(): shutil.copy(csv, DRIVE/'grid_v5.csv')
            if (OUT/'skipped.csv').exists():
                shutil.copy(OUT/'skipped.csv', DRIVE/'skipped.csv')
            print(f'[{stamp}] {mb:.1f} MB -> Drive  [safe]')
        else:
            print(f'[{stamp}] {mb:.1f} MB at {ZIP} -- DRIVE NOT MOUNTED, local only')
        json.dump({'n_img': n_img, 'n_row': n_row, 'at': stamp}, open(STATE, 'w'))

    if n_img != prev:
        try:
            from google.colab import files
            files.download(str(ZIP))
        except Exception as e:
            print(f'[{stamp}] browser download skipped ({e}) -- Drive copy is intact')